# 31 — Multi-Radius Satellite Scan

Queries Sentinel-5P catalogue coverage for 5, 10, 20, and 30 mile radii around Newhaven.

In [ ]:

CENTER_NAME = "Newhaven ERF"
CENTER_LAT = 50.80208
CENTER_LON = 0.04961
DATE_FROM = "2024-01-01"
DATE_TO   = "2025-12-31"
PRODUCT_KEY = "NO2"
TIMELINESS = "OFFL"
RADII_MILES = [5, 10, 20, 30]
MAX_SCENES_PER_RADIUS = 1000
OUTPUT_DIR = "outputs/newhaven_multi_radius"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": hourly_vars, "timezone": "UTC"}
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:

CDSE_USERNAME = os.getenv("CDSE_USERNAME") or os.getenv("CDSE_USER")
CDSE_PASSWORD = os.getenv("CDSE_PASSWORD")
assert CDSE_USERNAME, "Missing CDSE_USERNAME/CDSE_USER secret"
assert CDSE_PASSWORD, "Missing CDSE_PASSWORD secret"
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
ODATA_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
def get_cdse_token(username, password):
    r=requests.post(TOKEN_URL,data={"client_id":"cdse-public","username":username,"password":password,"grant_type":"password"},timeout=60)
    r.raise_for_status()
    return r.json()["access_token"]
token=get_cdse_token(CDSE_USERNAME,CDSE_PASSWORD)
headers={"Authorization":f"Bearer {token}"}
PRODUCT_PATTERNS={("NO2","OFFL"):"L2__NO2___",("NO2","NRTI"):"L2__NO2___",("SO2","OFFL"):"L2__SO2___",("SO2","NRTI"):"L2__SO2___",("CO","OFFL"):"L2__CO____",("CO","NRTI"):"L2__CO____"}
name_pat=PRODUCT_PATTERNS[(PRODUCT_KEY.upper(),TIMELINESS.upper())]


In [ ]:

rows=[]
for miles in RADII_MILES:
    km=miles*1.609344
    lat_deg=km/111.32
    lon_deg=km/(111.32*math.cos(math.radians(CENTER_LAT)))
    minx,maxx=CENTER_LON-lon_deg,CENTER_LON+lon_deg
    miny,maxy=CENTER_LAT-lat_deg,CENTER_LAT+lat_deg
    wkt=f"POLYGON(({minx} {miny},{maxx} {miny},{maxx} {maxy},{minx} {maxy},{minx} {miny}))"
    flt=("Collection/Name eq 'SENTINEL-5P' " + f"and ContentDate/Start ge {DATE_FROM}T00:00:00.000Z " + f"and ContentDate/Start le {DATE_TO}T23:59:59.999Z " + f"and OData.CSC.Intersects(area=geography'SRID=4326;{wkt}') " + f"and contains(Name,'{name_pat}') " + f"and contains(Name,'_{TIMELINESS.upper()}_')")
    params={"$filter":flt,"$top":str(MAX_SCENES_PER_RADIUS),"$orderby":"ContentDate/Start asc","$count":"true"}
    r=requests.get(ODATA_URL,params=params,headers=headers,timeout=180); r.raise_for_status()
    for p in r.json().get("value",[]):
        rows.append({"radius_miles":miles,"radius_km":km,"product_id":p.get("Id"),"name":p.get("Name"),"start_utc":p.get("ContentDate",{}).get("Start"),"end_utc":p.get("ContentDate",{}).get("End"),"published_utc":p.get("PublicationDate"),"online":p.get("Online"),"size_bytes":p.get("ContentLength")})
scene_df=pd.DataFrame(rows)
if not scene_df.empty:
    scene_df["start_utc"]=pd.to_datetime(scene_df["start_utc"],utc=True,errors="coerce")
    scene_df["date"]=scene_df["start_utc"].dt.date.astype("string")
scene_csv=Path(OUTPUT_DIR)/"satellite_scene_catalog.csv"
scene_df.to_csv(scene_csv,index=False)
display(scene_df.head(20))


In [ ]:

if not scene_df.empty:
    daily=scene_df.groupby(["radius_miles","date"],dropna=False).agg(scenes=("product_id","count"),first_overpass_utc=("start_utc","min")).reset_index()
    summary=scene_df.groupby("radius_miles",dropna=False).agg(total_scenes=("product_id","count"),days_with_scenes=("date","nunique"),mean_size_bytes=("size_bytes","mean")).reset_index().sort_values("radius_miles")
else:
    daily=pd.DataFrame(columns=["radius_miles","date","scenes","first_overpass_utc"])
    summary=pd.DataFrame(columns=["radius_miles","total_scenes","days_with_scenes","mean_size_bytes"])
daily_csv=Path(OUTPUT_DIR)/"satellite_daily_coverage.csv"
summary_csv=Path(OUTPUT_DIR)/"satellite_radius_summary.csv"
daily.to_csv(daily_csv,index=False); summary.to_csv(summary_csv,index=False)
display(summary)


In [ ]:

fig,ax=plt.subplots(figsize=(10,5))
if not summary.empty:
    ax.plot(summary["radius_miles"],summary["days_with_scenes"],marker="o",label="days_with_scenes")
    ax.plot(summary["radius_miles"],summary["total_scenes"],marker="o",label="total_scenes")
ax.set_title("Multi-radius satellite coverage summary"); ax.set_xlabel("Radius (miles)"); ax.set_ylabel("Count")
if not summary.empty: ax.legend()
fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/"satellite_radius_summary_plot.png"
fig.savefig(plot_path,dpi=150); plt.show()
manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"center_name":CENTER_NAME,"center_lat":CENTER_LAT,"center_lon":CENTER_LON,"date_from":DATE_FROM,"date_to":DATE_TO,"product_key":PRODUCT_KEY,"timeliness":TIMELINESS,"radii_miles":RADII_MILES,"rows_scene":int(len(scene_df)),"rows_daily":int(len(daily)),"rows_summary":int(len(summary)),"outputs":{"scene_csv":str(scene_csv),"daily_csv":str(daily_csv),"summary_csv":str(summary_csv),"plot_png":str(plot_path)}}
manifest_path=Path(OUTPUT_DIR)/"satellite_scan_manifest.json"
manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("Saved:",manifest_path)
